In [9]:
from pathlib import Path

import pandas as pd
from omegaconf import OmegaConf

from visgen.datasets import IRAVEN
from visgen.models import get_model
from visgen.models.resnet_mixer import RepresentationMixer

In [10]:
def num_pos(constellation_code: str) -> int:
    if constellation_code == "all":
        return max(pos[-1] for pos in IRAVEN.POSITION_MAP.values())
    constellation_name = IRAVEN.CONSTELLATION_CODE_TO_NAME[constellation_code]
    return len(IRAVEN.POSITION_MAP[constellation_name])


# Necesario para datasets IRAVEN con ${num_pos:...}
OmegaConf.register_new_resolver("num_pos", num_pos, replace=True)

In [28]:
BASE_CFG = Path("configs/base.yml")
EXPERIMENT_CFG = Path("configs/experiments/iid.yml")
DATASET_CFGS = sorted(Path("configs/datasets").glob("*.yml"))

MODEL_CFGS = {
    "resnet18": Path("configs/models/resnet18.yml"),
    "resnet18_mixer": Path("configs/models/resnet18_mixer.yml"),
    "resnet18_mixer_t64": Path("configs/models/resnet18_mixer_rp64_all_cases.yml"),
    "resnet18_mixer_t128": Path("configs/models/resnet18_mixer_rp128_all_cases.yml"),
    "resnet18_mixer_t256": Path("configs/models/resnet18_mixer_rp256_all_cases.yml"),
    "resnet18_mixer_alg": Path("configs/models/resnet18_algebraic_non_iid.yml"),
    "split_resnet": Path("configs/models/split.yml"),
    "split_resnet_mixer": Path("configs/models/split_resnet_mixer.yml"),
    "split_resnet_mixer_t64": Path("configs/models/split_resnet_mixer_s1_rp64_all_cases.yml"),
    "split_resnet_mixer_t128": Path("configs/models/split_resnet_mixer_s1_rp128_all_cases.yml"),
    "split_resnet_mixer_t256": Path("configs/models/split_resnet_mixer_s3_rp256_all_cases.yml"),
    "split_resnet_mixer_alg": Path("configs/models/split_resnet_mixer_alg_s1.yml"),
    "split_resnet_mixer_reduced_rep": Path("configs/models/split_resnet_mixer_reduced_rep.yml"),
    "ed": Path("configs/models/ed.yml"),
    "ed_mixer": Path("configs/models/ed_mixer.yml"),
    "ed_mixer_t64": Path("configs/models/ed_mixer_rp64.yml"),
    "ed_mixer_t128": Path("configs/models/ed_mixer_rp128.yml"),
    "ed_mixer_t256": Path("configs/models/ed_mixer_rp256.yml"),
}

MIXER_REP_PIECE_DIMS = (128, 256)

In [31]:

def build_cfg(dataset_cfg_path: Path, model_cfg_path: Path):
    cfg = OmegaConf.merge(
        OmegaConf.load(BASE_CFG),
        OmegaConf.load(EXPERIMENT_CFG),
        OmegaConf.load(dataset_cfg_path),
        OmegaConf.load(model_cfg_path),
    )

    cfg.device = "cpu"

    if not cfg.data.training.targets:
        all_targets = "_".join(att.name for att in cfg.data.training.attributes)
        cfg.data.training.targets = all_targets
        cfg.data.testing.targets = all_targets

    # --- defaults para ED ---
    if cfg.model.arch == "ed":
        if "preprocessing" not in cfg.model or cfg.model.preprocessing is None:
            cfg.model.preprocessing = []
        if "path" not in cfg.model or cfg.model.path is None:
            # ruta por dataset para evitar colisiones
            cfg.model.path = f"out/ed_{cfg.data.training.dataset}"

    return cfg

def count_parameters(module, trainable_only=False):
    params = module.parameters()
    if trainable_only:
        params = [p for p in params if p.requires_grad]
    return sum(p.numel() for p in params)


def summarize_model_dataset(model_name: str, dataset_cfg_path: Path, rep_piece_dim: int | None = None):
    cfg = build_cfg(dataset_cfg_path, MODEL_CFGS[model_name])

    if rep_piece_dim is not None:
        cfg.model.mixer.rep_piece_dim = rep_piece_dim

    model = get_model(cfg)

    total_params = count_parameters(model)
    trainable_params = count_parameters(model, trainable_only=True)

    mixer_params = 0
    mixer_trainable_params = 0
    if hasattr(model, "mixer") and model.mixer is not None:
        mixer_params = count_parameters(model.mixer)
        mixer_trainable_params = count_parameters(model.mixer, trainable_only=True)

    model_label = model_name
    if rep_piece_dim is not None:
        model_label = f"{model_name}_rep{rep_piece_dim}"

    return {
        "dataset_cfg": dataset_cfg_path.stem,
        "dataset": cfg.data.training.dataset,
        "modelo": model_label,
        "params_totales": total_params,
        "params_entrenables": trainable_params,
        "params_mixer": mixer_params,
        "params_mixer_entrenables": mixer_trainable_params,
        "params_training": total_params,
        "params_inferencia_sin_mixer": total_params - mixer_params,
        "mixer_rep_piece_dim": rep_piece_dim,
    }


In [32]:
# Tabla principal: modelo - dataset
rows = []

for dataset_cfg in DATASET_CFGS:
    for model_name in MODEL_CFGS:
        if model_name == "split_resnet_mixer_reduced_rep":
            for rep_piece_dim in MIXER_REP_PIECE_DIMS:
                rows.append(
                    summarize_model_dataset(
                        model_name,
                        dataset_cfg,
                        rep_piece_dim=rep_piece_dim,
                    )
                )
        else:
            rows.append(summarize_model_dataset(model_name, dataset_cfg))

model_dataset_df = pd.DataFrame(rows).sort_values(
    ["dataset", "modelo", "dataset_cfg"]
).reset_index(drop=True)
model_dataset_df


0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1 128 2
2 256 2
3 512 2
0 64 2
1

,dataset_cfg,dataset,modelo,params_totales,params_entrenables,params_mixer,params_mixer_entrenables,params_training,params_inferencia_sin_mixer
0,cars3d,cars3d,ed,33531072,33529536,0,0,33531072,33531072
1,cars3d_iid,cars3d,ed,33531072,33529536,0,0,33531072,33531072
2,cars3d_non_iid,cars3d,ed,33531072,33529536,0,0,33531072,33531072
3,cars3d,cars3d,resnet18,11284755,11284755,0,0,11284755,11284755
4,cars3d_iid,cars3d,resnet18,11284755,11284755,0,0,11284755,11284755
...,...,...,...,...,...,...,...,...,...
85,shapes3d_iid,shapes3d,split_resnet,12878834,12878834,0,0,12878834,12878834
86,shapes3d_non_iid,shapes3d,split_resnet,12878834,12878834,0,0,12878834,12878834
87,shapes3d,shapes3d,split_resnet_mixer,113646123,113646123,100738048,100738048,113646123,12908075
88,shapes3d_iid,shapes3d,split_resnet_mixer,113646123,113646123,100738048,100738048,113646123,12908075


In [37]:
# Tabla training vs inferencia (sin mixer)
training_vs_inference_df = model_dataset_df[
    [
        "dataset_cfg",
        "dataset",
        "modelo",
        "params_training",
        "params_inferencia_sin_mixer",
        "params_mixer",
    ]
].copy()

training_vs_inference_df["diferencia_training_vs_inferencia"] = (
    training_vs_inference_df["params_training"]
    - training_vs_inference_df["params_inferencia_sin_mixer"]
)

training_vs_inference_df

,dataset_cfg,dataset,modelo,params_training,params_inferencia_sin_mixer,params_mixer,diferencia_training_vs_inferencia
0,cars3d,cars3d,ed,33531072,33531072,0,0
1,cars3d_iid,cars3d,ed,33531072,33531072,0,0
2,cars3d_non_iid,cars3d,ed,33531072,33531072,0,0
3,cars3d,cars3d,resnet18,11284755,11284755,0,0
4,cars3d_iid,cars3d,resnet18,11284755,11284755,0,0
...,...,...,...,...,...,...,...
85,shapes3d_iid,shapes3d,split_resnet,12878834,12878834,0,0
86,shapes3d_non_iid,shapes3d,split_resnet,12878834,12878834,0,0
87,shapes3d,shapes3d,split_resnet_mixer,113646123,12908075,100738048,100738048
88,shapes3d_iid,shapes3d,split_resnet_mixer,113646123,12908075,100738048,100738048


In [55]:
# Mixer standalone usando la configuración de resnet18_mixer
cfg_mixer = build_cfg(DATASET_CFGS[0], MODEL_CFGS["resnet18_mixer"])
standalone_mixer = RepresentationMixer(
    emb_dim=cfg_mixer.model.emb_dim,
    num_layers=cfg_mixer.model.mixer.num_layers,
    num_heads=cfg_mixer.model.mixer.num_heads,
    dropout=cfg_mixer.model.mixer.dropout,
)

standalone_mixer_params = count_parameters(standalone_mixer)
standalone_mixer_trainable = count_parameters(standalone_mixer, trainable_only=True)

mixer_standalone_df = pd.DataFrame(
    [
        {
            "modelo": "representation_mixer_standalone",
            "params_totales": standalone_mixer_params,
            "params_entrenables": standalone_mixer_trainable,
            "params_mixer": standalone_mixer_params,
            "params_training": standalone_mixer_params,
            "params_inferencia_sin_mixer": 0,
        }
    ]
)

mixer_standalone_df

,modelo,params_totales,params_entrenables,params_mixer,params_training,params_inferencia_sin_mixer
0,representation_mixer_standalone,6307328,6307328,6307328,6307328,0


In [58]:
# Tabla final combinada
final_df = pd.concat(
    [
        model_dataset_df,
        mixer_standalone_df.assign(dataset_cfg="-", dataset="-"),
    ],
    ignore_index=True,
)

filter_df = final_df['dataset_cfg'].isin(['cars3d','dsprites','iraven','mpi3d','shapes3d','clevr'])
final_df = final_df[filter_df]

final_df = (
    final_df.groupby(["dataset", "modelo"], dropna=False)
      .mean(numeric_only=True)
      .reset_index()
)


final_df

,dataset,modelo,params_totales,params_entrenables,params_mixer,params_mixer_entrenables,params_training,params_inferencia_sin_mixer
0,cars3d,ed,33531072.0,33529536.0,0.0,0.0,33531072.0,33531072.0
1,cars3d,resnet18,11284755.0,11284755.0,0.0,0.0,11284755.0,11284755.0
2,cars3d,resnet18_mixer,17592083.0,17592083.0,6307328.0,6307328.0,17592083.0,11284755.0
3,cars3d,split_resnet,15252262.0,15252262.0,0.0,0.0,15252262.0,15252262.0
4,cars3d,split_resnet_mixer,46857209.0,46857209.0,31496704.0,31496704.0,46857209.0,15360505.0
5,clevr,ed,44708096.0,44706048.0,0.0,0.0,44708096.0,44708096.0
6,clevr,resnet18,11184720.0,11184720.0,0.0,0.0,11184720.0,11184720.0
7,clevr,resnet18_mixer,17492048.0,17492048.0,6307328.0,6307328.0,17492048.0,11184720.0
8,clevr,split_resnet,12033440.0,12033440.0,0.0,0.0,12033440.0,12033440.0
9,clevr,split_resnet_mixer,62424496.0,62424496.0,50382848.0,50382848.0,62424496.0,12041648.0


In [65]:
# Exportar tablas LaTeX separadas por dataset para que no queden demasiado largas

model_order = [
    "resnet18",
    "resnet18_mixer",
    "resnet18_mixer_alg",
    "resnet18_mixer_t64",
    "resnet18_mixer_t128",
    "resnet18_mixer_t256",
    "split_resnet",
    "split_resnet_mixer",
    "split_resnet_mixer_alg",
    "split_resnet_mixer_t64",
    "split_resnet_mixer_t128",
    "split_resnet_mixer_t256",
    "split_resnet_mixer_reduced_rep_rep128",
    "split_resnet_mixer_reduced_rep_rep256",
    "ed",
    "ed_mixer",
    "ed_mixer_t64",
    "ed_mixer_t128",
    "ed_mixer_t256",
]

df = final_df.copy()
df = df[
    df["dataset"].notna()
    & (df["dataset"] != "-")
    & df["modelo"].isin(model_order)
].copy()

MODEL_NAME_MAP = {
    "resnet18": "ResNet18",
    "resnet18_mixer": "ResNet18 + Mixer",
    "resnet18_mixer_alg": "ResNet18 + Mixer Algebraico",
    "resnet18_mixer_t64": "ResNet18 + Mixer Transformer (64)",
    "resnet18_mixer_t128": "ResNet18 + Mixer Transformer (128)",
    "resnet18_mixer_t256": "ResNet18 + Mixer Transformer (256)",
    "split_resnet": "AIN",
    "split_resnet_mixer": "AIN + Mixer",
    "split_resnet_mixer_alg": "AIN + Mixer Algebraico",
    "split_resnet_mixer_t64": "AIN + Mixer Transformer (64)",
    "split_resnet_mixer_t128": "AIN + Mixer Transformer (128)",
    "split_resnet_mixer_t256": "AIN + Mixer Transformer (256)",
    "split_resnet_mixer_reduced_rep_rep128": "AIN + Mixer ReducedRep (128)",
    "split_resnet_mixer_reduced_rep_rep256": "AIN + Mixer ReducedRep (256)",
    "ed": "ED",
    "ed_mixer": "ED + Mixer",
    "ed_mixer_t64": "ED + Mixer Transformer (64)",
    "ed_mixer_t128": "ED + Mixer Transformer (128)",
    "ed_mixer_t256": "ED + Mixer Transformer (256)",
}

DATASET_NAME_MAP = {
    "cars3d": "Cars3D",
    "dsprites": "dSprites",
    "shapes3d": "Shapes3D",
    "mpi3d": "MPI3D",
    "iraven": "I-RAVEN",
    "clevr": "CLEVR",
}

baseline = (
    df[df["modelo"] == "resnet18"][["dataset", "params_training", "params_inferencia_sin_mixer"]]
    .drop_duplicates("dataset")
    .rename(
        columns={
            "params_training": "resnet18_training",
            "params_inferencia_sin_mixer": "resnet18_inferencia",
        }
    )
)

tab = df.merge(baseline, on="dataset", how="left")
tab["pct_train_vs_resnet18"] = 100.0 * tab["params_training"] / tab["resnet18_training"]
tab["pct_inf_vs_resnet18"] = 100.0 * tab["params_inferencia_sin_mixer"] / tab["resnet18_inferencia"]

tab = tab.assign(
    dataset=tab["dataset"].map(DATASET_NAME_MAP).fillna(tab["dataset"]),
    modelo=tab["modelo"].map(MODEL_NAME_MAP).fillna(tab["modelo"]),
)

model_display_order = [MODEL_NAME_MAP.get(m, m) for m in model_order]
tab["modelo"] = pd.Categorical(tab["modelo"], categories=model_display_order, ordered=True)
tab = tab.sort_values(["dataset", "modelo"]).reset_index(drop=True)


def build_latex_table_for_dataset(dataset_name: str, dataset_tab: pd.DataFrame) -> str:
    lines = [
        r"\begin{table}[ht]",
        r"\centering",
        r"\small",
        r"\begin{tabular}{lcc}",
        r"\toprule",
        r"Modelo & \% de parámetros en training & \% de parámetros en inferencia \\",
        r"\midrule",
    ]

    for row in dataset_tab.itertuples(index=False):
        lines.append(
            f"{row.modelo} & {row.pct_train_vs_resnet18:.2f}\% & {row.pct_inf_vs_resnet18:.2f}\% \\")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            f"\caption{{{dataset_name}: porcentaje de parámetros respecto a ResNet18 (training e inferencia).}}",
            f"\label{{tab:parametros_vs_resnet18_{dataset_name.lower()}}}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

latex_tables_by_dataset = {
    dataset_name: build_latex_table_for_dataset(dataset_name, g)
    for dataset_name, g in tab.groupby("dataset", sort=True)
}

for dataset_name, latex_table in latex_tables_by_dataset.items():
    print(f"\n--- Tabla para {dataset_name} ---\n")
    print(latex_table)

     dataset              modelo  params_totales  params_entrenables  \
0     cars3d                  ed      33531072.0          33529536.0   
1     cars3d            resnet18      11284755.0          11284755.0   
2     cars3d      resnet18_mixer      17592083.0          17592083.0   
3     cars3d        split_resnet      15252262.0          15252262.0   
4     cars3d  split_resnet_mixer      46857209.0          46857209.0   
5      clevr                  ed      44708096.0          44706048.0   
6      clevr            resnet18      11184720.0          11184720.0   
7      clevr      resnet18_mixer      17492048.0          17492048.0   
8      clevr        split_resnet      12033440.0          12033440.0   
9      clevr  split_resnet_mixer      62424496.0          62424496.0   
10  dsprites                  ed      55853760.0          55851200.0   
11  dsprites            resnet18      11228209.0          11228209.0   
12  dsprites      resnet18_mixer      17535537.0          175355